# Compact Mamba versus temporal CNN: matched participant evaluation

Two separate experiments on a common, checksum-verified cohort. **10 outer participant folds, inner-only random search, and three training seeds.** Both families get equal candidate budgets and identical splits. This is a follow-up experiment motivated by earlier development, not a new external confirmation.

**Mamba reference:** width 64, two blocks, state 16, convolution 16, expansion 2, mean/std pooling and dropout 0.2. **CNN reference:** width 32, four residual blocks with two kernel-7 convolutions each, dilations 1/2/4/8, channel LayerNorm, mean/std pooling and dropout 0.2. Its per-position receptive field is 181 samples before global pooling.

No duplicated validation samples, oversampling, augmentation, positional embeddings or additional Mamba MLP. Threshold is fixed at 0.5 (ties PD). Original ambiguous identities are quarantined before splits. All clinical analyses are post-evaluation, never model inputs.

Enable Kaggle Internet for verified source/data setup and a CUDA GPU for training. Outputs stay in RAM unless explicitly exported. The old width-128 notebook remains a separate baseline.

In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, json, copy, zipfile

SOURCE_REV = '1672baba2e60d7752b10e8cf50d8d7a22d205d94'
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_MAMBA = False  # Set True only if the backend is absent; restart kernel afterwards.
MAMBA_PACKAGE = 'mamba-ssm==2.3.2.post1'  # Explicit version; GPU check remains mandatory.
roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for b in roots for p in (b, b/'paper_repository') if (p/'gait/comparison.py').is_file()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path(tempfile.gettempdir()) / ('gait-comparison-' + SOURCE_REV[:12])
    if not REPO_ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV], check=True)
REPO_ROOT = REPO_ROOT.resolve()
SOURCE_FILES = ('models.py','cross_validation.py','comparison_models.py','comparison.py','official_data.py','diagnostics.py')
for name in SOURCE_FILES:
    expected = subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+name], capture_output=True, check=True).stdout
    actual = (REPO_ROOT/'gait'/name).read_bytes()
    assert actual.replace(b'\r\n',b'\n') == expected.replace(b'\r\n',b'\n'), f'Changed pinned source: {name}'
sys.path.insert(0, str(REPO_ROOT))
if INSTALL_MAMBA:
    subprocess.run([sys.executable,'-m','pip','install',MAMBA_PACKAGE,'--no-build-isolation'], check=True)
print('Verified source:', SOURCE_REV)

## Freeze the search budget before training
Default **8 candidates per family × 10 folds × 3 seeds = 480 fits** (up to 28,800 epochs before early stopping). This can exceed one free session. Use `MAX_NEW_TASKS` to stop cleanly after a completed family/seed/fold task and explicitly export progress. An interrupted task restarts; no automatic mid-epoch checkpoint saving.

The reference is included, then random search spans: windows 500/1,000/2,000; training versus recording normalization; max versus mean/std pooling; dropout 0–0.3; AdamW LR 2.5e-5–3e-4 and weight decay 1e-4–1e-2 (log-uniform). Mamba width 32/64/128, 1–2 blocks, state 16/32, convolution 8/16/32. CNN width 32/48/64, 3–4 blocks, kernel 5/7. Shared trial-index preprocessing/optimizer draws are matched.

A cheaper declared experiment is four candidates and one seed (80 fits). Do not change the budget after examining outer results. Training seeds change initialization/shuffling, not the participant folds. Search seed controls candidate generation; split seed controls the shared outer/inner partitions.

Common eligibility uses at least 2,000 samples regardless of selected context, preserving the previously curated cohort. Recording normalization is an **offline** complete-recording transform. Class weights use inner-training window counts only; participants with more windows still influence training more.

In [ ]:
from dataclasses import asdict
from collections import Counter
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from html import escape
from gait import cross_validation as cv
from gait import comparison as cmp
from gait import official_data as official
from gait import diagnostics as diagnostics_module
assert Path(cmp.__file__).resolve().is_relative_to(REPO_ROOT), 'Restart kernel: stale gait imports'

CACHE_DIR = Path(tempfile.gettempdir()) / 'gaitpdb-1.0.0-verified-cache'
ALLOW_DOWNLOAD = True  # Input cache only; attached Kaggle data remain untouched.
TRAINING_SEEDS = (42, 123, 2026)
SEARCH_SEED = 90210
CANDIDATES_PER_FAMILY = 8
COMMON_MIN_SAMPLES = 2000
CFG = cv.Protocol(seq_len=1000, overlap=0.0, outer_folds=10, inner_fraction=0.2,
                  split_seed=42, training_seeds=TRAINING_SEEDS, batch_size=32,
                  max_epochs=60, patience=10, scheduler_patience=4, scheduler_factor=0.5,
                  min_lr=1e-7, bootstrap_samples=2000, bootstrap_seed=7301, num_workers=0)
CANDIDATES = cmp.sample_candidates(CANDIDATES_PER_FAMILY, SEARCH_SEED)
cmp.validate_search(CFG, CANDIDATES)
RUN_EXPERIMENT = False
MAX_NEW_TASKS = 2  # Two tasks = one outer fold, both families. None runs all remaining tasks.
results = globals().get('results', {})  # Never silently erase completed-fold progress.
PLAN = json.loads(json.dumps({'protocol':asdict(CFG),'candidates':CANDIDATES,
                             'search_seed':SEARCH_SEED,'common_min_samples':COMMON_MIN_SAMPLES}))
print(json.dumps(PLAN, indent=2))
print('Planned fits:', 2 * CFG.outer_folds * len(TRAINING_SEEDS) * CANDIDATES_PER_FAMILY)
print(cv.runtime_info())

def show_table(rows, columns=None, title='', limit=None):
    rows = list(rows)
    if title: print(title, '| rows:', len(rows))
    if not rows: print('No rows'); return
    columns = columns or list(dict.fromkeys(k for r in rows for k in r))
    def format_value(v):
        if v is None: return 'Unknown / not applicable'
        return f'{v:.4f}' if isinstance(v, float) else str(v)
    shown = rows if limit is None else rows[:limit]
    html = '<div style="max-height:480px;overflow:auto"><table><thead><tr>'
    html += ''.join('<th>'+escape(c)+'</th>' for c in columns) + '</tr></thead><tbody>'
    html += ''.join('<tr>'+''.join('<td>'+escape(format_value(r.get(c)))+'</td>' for c in columns)+'</tr>' for r in shown)
    display(HTML(html+'</tbody></table></div>'))
    if limit is not None and len(rows)>limit: print('Display limited; complete table remains in RAM and optional export.')

## Verify the full cohort and clinical sources
Pin the official manifest checksum, verify all 306 originals, then exclude **every recording** belonging to an identity implicated in cross-ID force duplicates. Known groups: GaPt17/GaPt18 and JuCo26/JuPt01; the second has conflicting diagnoses. Within-identity copies are counted once. No model performance informs exclusions.

Clinical TXT and HTML are independently checksum-verified and cross-checked by exact ID, study, diagnosis and HY. The TXT's later merged fields are not used. The malformed source ID `Juc010` is logged, not guessed. Missing/control HY is never inferred. Unknown clinical data do not remove a participant from model evaluation.

In [ ]:
minimum = max(COMMON_MIN_SAMPLES, max(c['seq_len'] for cs in CANDIDATES.values() for c in cs))
records, audit, provenance = official.load_verified_cohort(CACHE_DIR, minimum, ALLOW_DOWNLOAD)
clinical, clinical_provenance = official.load_verified_clinical(CACHE_DIR, ALLOW_DOWNLOAD)
splits = cv.make_splits(records, CFG)
subject_labels = {r['subject']:r['label'] for r in records}
print('Curation:', provenance)
print('Audit statuses:', dict(Counter(r['status'] for r in audit)))
print('Participant counts:', dict(Counter(subject_labels.values())))
print('Retained recordings:', len(records))
print('Unmatched clinical IDs:', sorted(set(subject_labels)-set(clinical)))
print('Clinical source issues:', clinical_provenance['issues'])
for length in sorted({c['seq_len'] for cs in CANDIDATES.values() for c in cs}):
    print('Context',length,'samples ->',len(cv.make_windows(records,length,0)),'windows')
show_table([{'fold':s['fold'], **{part:dict(Counter(subject_labels[p] for p in s[part])) for part in ('train','validation','test')}} for s in splits], title='Shared fold assignments (0=Control, 1=PD)')
show_table([{'diagnosis':label,'hy_status':status,'n':n} for (label,status),n in Counter((subject_labels[s], clinical.get(s,{}).get('hy_status','unmatched')) for s in subject_labels).items()], title='Clinical coverage before training')
FROZEN_PLAN = copy.deepcopy(PLAN)

## Actual backend check for every candidate
The check executes real forward/backward operations, preserving Mamba initialization and verifying finite gradients. It is not a training result. CNN is also tested on the same GPU. Listed receptive fields apply only to CNN features before pooling; Mamba has a recurrent temporal path, not a finite CNN receptive-field number.

The smoke batch is two; a training batch of 32 may require more memory. If training runs out of memory, start a new fixed plan with a smaller batch for **both** families; do not silently skip expensive candidates.

In [ ]:
GPU_READY = False
if not torch.cuda.is_available():
    print('CUDA unavailable. Audit and diagnostics can run, but this experiment requires Kaggle CUDA.')
else:
    smoke_report = cmp.smoke_candidates(CANDIDATES, batch_size=2)
    show_table(smoke_report, title='Real model forward/backward checks passed')
    GPU_PLAN = copy.deepcopy(PLAN)
    GPU_READY = True

## Optional resume from an explicitly exported bundle
Leave the path empty for a fresh run. Upload your own previously exported ZIP as a Kaggle input to continue in a new session. Loading reads only JSON, not pickled models or executable code. Never merge results from different plans. Source, candidate list, cohort, runtime and completed-task consistency are checked when training resumes. Clinical metadata are reverified independently.

In [ ]:
RESUME_ZIP = ''  # e.g. '/kaggle/input/my-run-progress/comparison_progress.zip'
if RESUME_ZIP:
    if results.get('runs'): raise RuntimeError('Results already in RAM; do not overwrite them')
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        results = json.loads(archive.read('results.json'))
    print('Loaded completed tasks:',len(results.get('runs',[])),'| validated against the current plan at resume')
else:
    print('No progress bundle loaded.')

## Run or continue the frozen experiment
The scheduler, stopping, checkpoint and hyperparameter selection use **inner participant log loss** only. Every family has its own winner; both are tested, and no family is selected from outer scores. The selected checkpoint is tested without refitting. Outer scores and error plots are withheld until all planned tasks finish.

A completed task contains all candidate histories, selected inner predictions, test windows/participants, normalizer, parameter counts, training times and peak GPU allocation. Progress prints one line per candidate, reducing output truncation. Times are workload measurements, not a controlled inference benchmark. No failed candidate is silently skipped.

In [ ]:
if RUN_EXPERIMENT:
    current = json.loads(json.dumps({'protocol':asdict(CFG),'candidates':CANDIDATES,
                                    'search_seed':SEARCH_SEED,'common_min_samples':COMMON_MIN_SAMPLES}))
    assert current == FROZEN_PLAN, 'Configuration changed after audit; freeze a new plan first'
    assert GPU_READY and GPU_PLAN == current, 'Run the GPU check for this exact plan first'
    cmp.run_comparison(records, audit, provenance, CFG, CANDIDATES, progress=results,
                       max_new_tasks=MAX_NEW_TASKS, verbose=False)
    results['source_revision'] = SOURCE_REV
    results['search_seed'] = SEARCH_SEED
    results['clinical_provenance'] = clinical_provenance
    print('Completed tasks:',len(results['runs']),'of',2*CFG.outer_folds*len(CFG.training_seeds))
    print('Full experiment complete:',results['complete'])
else:
    print('Training disabled. Set RUN_EXPERIMENT=True after audit and GPU check. Results are preserved.')

## Explicit export (available between chunks)
Set `SAVE_RESULTS=True` only when you want to preserve completed tasks on disk. Each export gets a fresh timestamped filename; no overwrite, raw signals or automatic downloads. In-progress task weights are not saved. Before a session ends, export and download through Kaggle's file browser; RAM-only results do not survive a reset. This cell can be rerun after diagnostics to include all tables.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    from datetime import datetime, timezone
    if not results.get('runs'): raise RuntimeError('No completed task available')
    results['source_revision'] = SOURCE_REV
    results['search_seed'] = SEARCH_SEED
    results['clinical_provenance'] = clinical_provenance
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    export_path = Path('/kaggle/working') / f'comparison_progress_{stamp}.zip'
    print('Saved:',cv.export_results(results,export_path))
else:
    print('No result files written. Progress remains in RAM.')

## Complete OOF metrics and paired comparison
Each eligible participant is tested exactly once per model/seed. Pooled metrics are primary, not an unweighted average of unequal fold sizes. Individual bootstrap intervals resample participants within diagnosis, conditional on fitted OOF predictions and class counts; they exclude refitting and new-split uncertainty. Fold SD is not a CI. Repeated seed rows are never treated as independent extra people.

Paired differences resample the same participants for both families and every seed, then average the seed-specific metric differences. This is **not** a prediction ensemble. Intervals are descriptive and conditional, not proof of external superiority. Do not select the best seed.

In [ ]:
if not results.get('complete'):
    print('Finish all planned tasks before inspecting outer results.')
else:
    diagnostics = diagnostics_module.build_diagnostics(results, clinical)
    results['diagnostics'] = diagnostics
    results['clinical_provenance'] = clinical_provenance
    metric_rows = []
    for family, summaries in results['summaries'].items():
        for seed, summary in summaries.items():
            for name, interval in summary['confidence_intervals'].items():
                metric_rows.append({'family':family,'seed':seed,'metric':name,**interval})
    show_table(metric_rows, title='Participant OOF metrics and 95% bootstrap intervals')
    show_table([{'family':f,'seed':s,**v['participant_metrics'],'fold_accuracy_sd':v['fold_accuracy_std']} for f,ss in results['summaries'].items() for s,v in ss.items()], title='Counts, log loss and fold SD')
    show_table([{'family':f,'seed':s,**v['window_metrics']} for f,ss in results['summaries'].items() for s,v in ss.items()], title='Secondary pooled window metrics')
    across = []
    for family, summaries in results['summaries'].items():
        for metric in ('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc'):
            values = [s['participant_metrics'][metric] for s in summaries.values()]
            across.append({'family':family,'metric':metric,'seed_mean':float(np.mean(values)),
                           'seed_sample_sd':float(np.std(values,ddof=1)) if len(values)>1 else None,'n_seeds':len(values)})
    show_table(across, title='Training-seed variability on the same folds')
    show_table(diagnostics['paired_comparison'], title='Paired Mamba minus CNN differences: seed-averaged metrics')

## Participant errors, cohort patterns and clinical severity
These are exploratory descriptions, not causal explanations. Study/diagnosis and HY tables include their actual sample sizes. Only verified metadata determine HY; absent or conflicting values stay Unknown. Control HY=0 is never inferred. A single-class subgroup has no defined ROC-AUC. Wilson intervals for subgroup correctness are pointwise, not multiplicity-adjusted confirmatory comparisons.

PD stage sensitivity counts correctly detected PD participants at each stage; it is not severity-classification accuracy. Small stage/study cells can be highly uncertain. `confident_error` means wrong predicted class with probability at least 0.8; probabilities are not assumed calibrated. All errors remain inspectable, not only confident ones.

Inner-validation high-loss rows come from selected checkpoints and may repeat a person across folds; do not combine them as independent people or confuse them with outer tests. Their individual NLL contributions can explain why a small number of errors affect validation curves.

In [ ]:
if results.get('complete') and 'diagnostics' in results:
    diagnostics = results['diagnostics']
    show_table(diagnostics['participant_errors'], ['family','seed','fold','subject','study','diagnosis','probability','prediction','correct','recordings','windows','hy','hy_status','age','confidence','nll'], title='Complete participant-level outer error table')
    show_table(diagnostics['by_study'], title='Outer results by contributing study')
    show_table(diagnostics['by_study_diagnosis'], title='Outer results by study and diagnosis')
    show_table(diagnostics['by_fold'], title='Outer results by fold')
    show_table(diagnostics['pd_severity'], title='PD sensitivity by verified Hoehn–Yahr stage (including unknown/absent)')
    show_table(diagnostics['pd_severity_study'], title='PD sensitivity by study and stage')
    show_table(diagnostics['metadata_coverage'], title='Metadata coverage and unresolved joins')
    show_table(diagnostics['confidently_wrong'], title='Confidently wrong outer participants, ranked by NLL')
    show_table(sorted(diagnostics['participant_errors'], key=lambda r:-r['nll']), title='All outer participants ranked by individual NLL', limit=30)
    show_table(diagnostics['inner_high_loss'], title='Selected inner-validation checkpoint NLL contributors (not independent across folds)', limit=30)
    show_table(sorted(diagnostics['seed_stability'], key=lambda r:(-r['error_seeds'],r['family'],r['subject'])), title='Participant consistency across training seeds')
else:
    print('Run the complete-result metrics cell first.')

In [ ]:
if results.get('complete'):
    selection_rows, trial_rows = [], []
    for r in results['runs']:
        selected = r['trials'][r['selected_candidate']]
        selection_rows.append({'family':r['family'],'seed':r['seed'],'fold':r['fold'],**selected['candidate'],
                               'best_epoch':selected['best_epoch'],'parameters':selected['parameters'],
                               'inner_nll':selected['inner_metrics']['log_loss']})
        for i,t in enumerate(r['trials']):
            trial_rows.append({'family':r['family'],'seed':r['seed'],'fold':r['fold'],**t['candidate'],
                               'selected':i==r['selected_candidate'],'best_epoch':t['best_epoch'],
                               'inner_nll':t['inner_metrics']['log_loss'],'parameters':t['parameters'],
                               'training_seconds':t['seconds'],'peak_allocated_bytes':t['peak_allocated_bytes'],
                               'receptive_field_samples':t['receptive_field_samples']})
    results['selection_table'] = selection_rows
    results['trial_table'] = trial_rows
    show_table(selection_rows, title='Selected settings in every model/seed/fold')
    show_table(trial_rows, title='All search trials and measured resource use', limit=40)
    print('Total candidate-training hours:',sum(r['training_seconds'] for r in trial_rows)/3600)

In [ ]:
# Display only; no plot files, interpolated histories, or artificial epoch padding.
if results.get('complete'):
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
    for seed in results['protocol']['training_seeds']:
        fig, axes = plt.subplots(2, 2, figsize=(11, 9))
        for index,family in enumerate(('mamba','cnn')):
            rows = [p for r in results['runs'] if r['family']==family and r['seed']==seed for p in r['test_participants']]
            y = np.array([p['label'] for p in rows]); p = np.array([p['probability'] for p in rows])
            ConfusionMatrixDisplay(confusion_matrix(y,p>=.5,labels=[0,1]),display_labels=['Control','PD']).plot(ax=axes[index,0],colorbar=False)
            axes[index,0].set_title(f'{family} | seed {seed} | participant OOF')
            fpr,tpr,_ = roc_curve(y,p)
            axes[index,1].plot(fpr,tpr,label=f'AUC={roc_auc_score(y,p):.3f}')
            axes[index,1].plot([0,1],[0,1],'--',color='gray'); axes[index,1].legend()
            axes[index,1].set(xlabel='False positive rate',ylabel='True positive rate',title=family+' participant ROC')
        fig.tight_layout(); plt.show()
        fig, axes = plt.subplots(2,2,figsize=(12,8))
        for index,family in enumerate(('mamba','cnn')):
            for r in results['runs']:
                if r['family']!=family or r['seed']!=seed: continue
                history=r['trials'][r['selected_candidate']]['history']
                epochs=[h['epoch'] for h in history]
                axes[index,0].plot(epochs,[h['train_loss'] for h in history],alpha=.7,label=f"Fold {r['fold']}")
                axes[index,1].plot(epochs,[h['validation_log_loss'] for h in history],alpha=.7)
            axes[index,0].set(title=family+' inner training',xlabel='Observed epoch',ylabel='Weighted window CE')
            axes[index,1].set(title=family+' inner validation',xlabel='Observed epoch',ylabel='Participant NLL')
            axes[index,0].legend(ncol=2,fontsize=8)
        fig.suptitle(f'Selected candidates | training seed {seed}'); fig.tight_layout(); plt.show()
    if 'diagnostics' in results:
        stability=results['diagnostics']['seed_stability']
        fig,ax=plt.subplots(figsize=(8,4))
        for family in ('mamba','cnn'):
            counts=Counter(r['error_seeds'] for r in stability if r['family']==family)
            x=np.arange(len(results['protocol']['training_seeds'])+1)
            ax.plot(x,[counts.get(int(i),0) for i in x],marker='o',label=family)
        ax.set(xlabel='Training seeds that misclassified a participant',ylabel='Participants',title='Persistent versus variable errors')
        ax.legend(); plt.show()

## Preserve and interpret the result
Rerun the explicit export cell after diagnostics to include the full tables. Report both models, all declared seeds, the search budget, actual cohort exclusions and source/environment provenance. Do not present post-hoc HY/study patterns as validated biomarkers, or conditional bootstrap intervals as uncertainty across hospitals or refitted datasets.

If follow-up changes are motivated by these outer errors, label them exploratory and predeclare the next experiment. More inner candidates can overfit a 29-person inner holdout; full inner cross-validation and external validation are higher-cost extensions.